## Evaluation Notebook for Intelligent Systems Assignment 2

In [ ]:
from decision_tree import ID3, CAR_COLUMNS, OUTCOME_CLASS, classify
import polars as pl
import numpy as np
from helpers import Metric, Metrics

In [2]:
df_csv = pl.read_csv("./dt-car.csv", schema_overrides=CAR_COLUMNS)

In [3]:
partition = 0.8

temp_df = df_csv.with_columns(pl.lit(np.random.rand(df_csv.height)).alias("random_no"))
test = temp_df.filter(pl.col("random_no") > partition).drop("random_no")
train = temp_df.filter(pl.col("random_no") <= partition).drop("random_no")

print(f"test = {test.height}, train = {train.height}")

test = 358, train = 1370


In [4]:
tree = ID3(train, set(train.columns) - {OUTCOME_CLASS}, train)

$\text{Accuracy} = \frac{TP + TN}{FN+FP + TP + TN}$

$\text{Precision} = \frac{TP}{TP+FP}$

$\text{Recall} = \frac{TP}{TP+FN}$

$\text{F1 Score} = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}} {\text{Precision} + \text{Recall}}$

In [5]:
classes = test[OUTCOME_CLASS].unique()
m = Metrics(list(classes))

for row in test.rows(named=True):
    model_prediction = classify(row, tree)
    ground_truth = row["class"]

    m.update(model_prediction, ground_truth)


In [6]:
m.print_summary()

                     precision       recall          f1-score        support        
--------------------------------------------------------------------------------
unacc                0.9683          0.9683          0.9683          252            
acc                  0.9114          0.8675          0.8889          83             
good                 0.7333          0.9167          0.8148          12             
vgood                0.7500          0.8182          0.7826          11             
--------------------------------------------------------------------------------
macro avg            0.8407          0.8926          0.8636          358            
weighted avg         0.9405          0.9385          0.9390          358            
--------------------------------------------------------------------------------

accuracy             0.9385


In [34]:
partitions = np.arange(0, 1.01, 0.01)
training_sizes = [int(i * train.height) for i in partitions]

accuracies = []

for size in training_sizes:
    training_partition = train[0:size]
    tree = ID3(training_partition, set(training_partition.columns) - {OUTCOME_CLASS}, training_partition)

    correct = 0
    for row in test.rows(named=True):
        model_prediction = classify(row, tree)
        ground_truth = row["class"]
        if model_prediction == ground_truth: correct += 1
    acc = correct/test.height

    print(f"tree for {size} complete (acc={acc}%)")

    accuracies.append(acc)

tree for 0 complete (acc=0.0%)
tree for 13 complete (acc=0.7039106145251397%)
tree for 27 complete (acc=0.7039106145251397%)
tree for 41 complete (acc=0.7039106145251397%)
tree for 54 complete (acc=0.7039106145251397%)
tree for 68 complete (acc=0.7039106145251397%)
tree for 82 complete (acc=0.7039106145251397%)
tree for 95 complete (acc=0.7039106145251397%)
tree for 109 complete (acc=0.7039106145251397%)
tree for 123 complete (acc=0.7039106145251397%)
tree for 137 complete (acc=0.7039106145251397%)
tree for 150 complete (acc=0.7039106145251397%)
tree for 164 complete (acc=0.7039106145251397%)
tree for 178 complete (acc=0.7039106145251397%)
tree for 191 complete (acc=0.729050279329609%)
tree for 205 complete (acc=0.7458100558659218%)
tree for 219 complete (acc=0.7486033519553073%)
tree for 232 complete (acc=0.7486033519553073%)
tree for 246 complete (acc=0.7541899441340782%)
tree for 260 complete (acc=0.7541899441340782%)
tree for 274 complete (acc=0.7541899441340782%)
tree for 287 comp

In [ ]:
import plotly.express as px

fig = px.line(x=training_sizes, y=accuracies)
fig.show()